# Parameter Sensitivity

Dedicated one-parameter sensitivity workflow. The default values reproduce the configuration formerly embedded in `run_model.ipynb`.

In [ ]:
# Setup
%load_ext autoreload
%autoreload 2

from config import MACRO_COLUMNS, SCENARIO_PRESETS
from src.notebook_workflow import NotebookRunConfig, build_country_config, prepare_data
from src.sensitivity import run_parameter_sensitivity
from src.visual_helpers import plot_sensitivity, plot_sensitivity_summary


## Inputs

In [ ]:
RUN_SENSITIVITY = False
SCENARIO_NAME = "calibrated_consumption"
run_config = NotebookRunConfig(seed=232, t_max=150, country_iso3="FRA", force_rebuild_data=True)
parameter_path = "households.functions.consumption.parameters['long_run_intercept']"
parameter_values = [-0.1, -0.075, -0.05, 0, 0.05, 0.075, 0.1]
seeds = [18, 21, 26, 32, 377, 443, 435, 427, 144, 254, 543, 656, 278, 59]
N_JOBS = -1
BATCH_SIZE = 4


## Prepare shared inputs

In [ ]:
prepared = prepare_data(run_config)
COUNTRY = prepared.cfg.country_iso3
country_configurations = build_country_config(
    data=prepared.data,
    config=run_config,
    overrides=SCENARIO_PRESETS[SCENARIO_NAME],
)


## Run

In [ ]:
sensitivity = None
if RUN_SENSITIVITY:
    sensitivity = run_parameter_sensitivity(
        datawrapper=prepared.data,
        country_configurations=country_configurations,
        country_code=COUNTRY,
        parameter_path=parameter_path,
        parameter_values=parameter_values,
        seeds=seeds,
        t_max=prepared.cfg.t_max,
        n_jobs=N_JOBS,
        backend="loky",
        batch_size=BATCH_SIZE,
    )
else:
    print("Set RUN_SENSITIVITY = True to run the experiment.")


## Plots

In [ ]:
if sensitivity is not None:
    plot_sensitivity_summary(sensitivity, cols=list(MACRO_COLUMNS))
    plot_sensitivity(sensitivity, cols=list(MACRO_COLUMNS))
